# Early FTUE analysis

**Purpose:** 

**Data range:** 

---



## Key Findings

**Retention:**
- B.Post-FTUE revamp shows marginally lower D1 retention on both platforms (−0.5 to −0.6pp across 22 cohorts). The gap grows over time — by D21 B.Post Android trails A.Pre by ~4pp (8.8% vs 12.8%). *("Retention rate by FTUE group" chart)*
- Android organic retention gap is the starkest signal: D14 drops from 12.2% (A.Pre) to 9.2% (B.Post, −3.0pp); D21 drops from 9.1% to 5.3% (−3.8pp). *("Retention rate for Organics" chart)*
- iOS organic shows a partial cross-over: B.Post trails at D1–D3 but leads at D14 (12.0% vs 9.9%, +2.1pp), suggesting the revamp may be helping iOS mid-term retention while hurting Android. *("Retention rate for Organics" chart)*
- B.Post cohort sizes are ~30% larger than A.Pre at all checkpoints, reflecting install volume growth over the study period — not a retention improvement. *("Cohort sizes" chart)*

**Player Level Progression:**
- Median (P50) level reached per day since install is nearly identical between A.old and B.new through D14 on both platforms. Android B.new shows a 2-level lag only at D21, likely a survivor-bias artefact from weaker retention rather than slower pacing. *("Percentile comparison" band chart)*
- P10–P90 band shapes closely mirror each other across groups and platforms — the revamp has not created meaningfully different engagement depth within the active player population. *("Player level P10/P50/P90" band chart)*

In [ ]:
# hide-output
# Import libraries and initialise the BigQuery connector

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np
import plotly.graph_objects as go

bqc = BigQueryConnector()

## Aux functions

In [ ]:
# hide-output
# Helper functions for weighted progression, percentile calculation, and level visualisations
from aux_functions import (
    compute_weighted_progression,
    weighted_quantiles,
    add_event_annotations,
    add_median_lines,
    plot_percentile_comparison,
)

## Get data

### Player level and game day

In [ ]:
# Compute symmetric A/B date windows of equal length anchored on 2026-06-01 (FTUE launch date)

import datetime as dt

new_ftue_date = dt.datetime(2026, 6, 1)
days_from_start = (dt.datetime.today() - new_ftue_date).days
start_date1 = new_ftue_date - dt.timedelta(days=days_from_start)
end_date1 = new_ftue_date-dt.timedelta(days=1)
start_date2 = new_ftue_date
end_date2 = dt.datetime.today()-dt.timedelta(days=1)

# print all dates
print(f"Start Date 1: {start_date1.strftime('%Y-%m-%d')}")
print(f"End Date 1: {end_date1.strftime('%Y-%m-%d')}")
print(f"Start Date 2: {start_date2.strftime('%Y-%m-%d')}")
print(f"End Date 2: {end_date2.strftime('%Y-%m-%d')}")

In [ ]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
refresh_data = True

In [ ]:
# hide-output
# Estimate query cost for player level + game day SQL before executing

query_location = './sql/playerlevel.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

In [ ]:
# hide-output
# Fetch player level + game day data from BigQuery or load from local pickle cache
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    data = bqc.get(query='./sql/playerlevel.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playprogression.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playprogression.pkl')

In [ ]:
# hide-output
# Preview raw player level and game day data
data

### Retention

In [ ]:
# hide-output
# Estimate query cost for per-install-date retention SQL
query_location = './sql/retention.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

In [ ]:
# hide-output
# Fetch per-install-date retention data from BigQuery or load from local pickle cache
retention_data = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data = bqc.get(query='./sql/retention.sql', is_path=True, query_parameters=parameters)
    retention_data.to_pickle('./data/retention.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data = pd.read_pickle('./data/retention.pkl')

In [ ]:
# hide-output
# Sort retention data and spot-check Android rows
retention_data.sort_values(['install_dt', 'dx','platform'], inplace=True)
retention_data[retention_data['platform'] == 'AND']

In [ ]:
# hide-output
# Estimate query cost for FTUE-split retention SQL (all users)
query_location = './sql/retentiontotal.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

In [ ]:
# hide-output
# Fetch FTUE-split retention (all users) from BigQuery or load from local pickle cache
retention_data_total = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data_total = bqc.get(query='./sql/retentiontotal.sql', is_path=True, query_parameters=parameters)
    retention_data_total.to_pickle('./data/retentiontotal.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total = pd.read_pickle('./data/retentiontotal.pkl')

In [ ]:
# hide-output
# Sort FTUE retention data and spot-check at D14
retention_data_total.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total[retention_data_total['dx'] == 14]

In [ ]:
# hide-output
# Estimate query cost for FTUE-split organic-only retention SQL
query_location = './sql/retentiontotalNA.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

In [ ]:
# hide-output
# Fetch FTUE-split organic-only retention from BigQuery or load from local pickle cache
retention_data_total_na = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache
if refresh_data:
    retention_data_total_na = bqc.get(query='./sql/retentiontotalNA.sql', is_path=True, query_parameters=parameters)
    retention_data_total_na.to_pickle('./data/retentiontotalNA.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total_na = pd.read_pickle('./data/retentiontotalNA.pkl')

In [ ]:
# hide-output
# Sort and preview organic-only FTUE retention data
retention_data_total_na.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total_na

## Process data

In [ ]:
# hide-output
# Assign FTUE flag, cap days_since_install to match B.new window, and drop immature cohort rows
dt_mode = 'install_dt'

data['install_dt'] = data[dt_mode]

data['CPE_flag'] = ['Y' if x == 'CPE' else 'N' for x in data['acquisition_type']]

data['FTUE_flag'] = ['B.new' if x >='0.76.0' else 'A.old' for x in data['install_build_version']]

# Making comparison fair
max_dayx_B_new = (pd.to_datetime('today') - pd.to_datetime('2026-06-01')).days
data = data[~(data['days_since_install'] > max_dayx_B_new)]

data.loc[:,'dummy'] = 'dummy'

# Require each cohort to have had enough calendar time to be meaningful:
# weekly cohorts need 7 days, monthly cohorts need 30.
min_days_since_install = 0
if dt_mode == 'install_dt_week':
    min_days_since_install = 7
elif dt_mode == 'install_dt_month':
    min_days_since_install = 30 

# Drop rows where the player's most recent observed day falls within the cohort's maturity window.
# This prevents partially-observed cohorts from pulling down progression averages.
data = data[data['days_since_install'] <= (pd.to_datetime('today') - pd.to_datetime(data['install_dt'])).dt.days - min_days_since_install]

data

In [ ]:
# hide-output
# Sanity-check unique user counts per FTUE group
test = data.groupby(['FTUE_flag']).agg(
    users=('user_id', 'nunique')
).reset_index()

test

## Retention

Retention curve: the share of a cohort's total users who were active on each calendar day since install. Plotted on a **log scale** so that differences between cohorts remain visible at longer horizons where absolute percentages are very small. Apr 2024 D1 retention was ~55%; Apr 2026 has declined to ~51%.

In [ ]:
# hide-output
# Add combined dx_platform column for the per-install-date retention line chart
retention_data['combined_dimension'] = retention_data['dx'].astype(str) + '_' + retention_data['platform']
retention_data

In [ ]:
# Per-install-date retention rate over time by dx/platform (figure disabled — used for investigation only)
fig = px.line(retention_data[retention_data['dx'] !=0], 
              x='install_dt', 
              y='retention_rate',
              color='combined_dimension',
              title='Retention rate',
              facet_row='platform',
              width=1200,
              height=800,
              hover_data={'install_dt': True, 'retained_size': True},)

#fig.show()

In [ ]:
# hide-output
# Preview FTUE-split retention totals (all users)
retention_data_total

### Cohort sizes

In [ ]:
# Bar chart: cohort sizes at each retention checkpoint by FTUE group and platform
df_plot = retention_data_total[retention_data_total['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='cohort_size',
    color='FTUE_flag',
    text='cohort_size',
    title='Cohort sizes',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True, 'num_cohorts': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:0}', textposition='outside')
fig.update_layout(
    #yaxis_tickformat='.0%',
    #yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

### Retantion rate 

In [ ]:
# Bar chart: D1–D21 retention rates by FTUE group and platform (all users)
df_plot = retention_data_total[retention_data_total['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='retention_rate',
    color='FTUE_flag',
    text='retention_rate',
    title='Retention rate by FTUE group (95% CI based on cohort size)',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True, 'num_cohorts': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig.update_layout(
    yaxis_tickformat='.0%',
    yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

### Retention rate for Organics

In [ ]:
# Bar chart: D1–D21 retention rates by FTUE group and platform (organic / non-attributed only)
df_plot = retention_data_total_na[retention_data_total_na['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='retention_rate',
    color='FTUE_flag',
    text='retention_rate',
    title='Retention rate by FTUE group (Only non-attributed)',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True, 'num_cohorts': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig.update_layout(
    yaxis_tickformat='.0%',
    yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

## Player max level distribution

In [ ]:
# Preview player data
data

In [ ]:
# hide-output
# Build level funnel with P10/P50/P90 percentiles per FTUE group, platform, and day since install
pl_ftue_funnel_agg = data.groupby(['max_level','FTUE_flag','platform', 'days_since_install']).agg(
    users=('user_id', 'nunique')
).reset_index()

pl_ftue_funnel_total_agg = pl_ftue_funnel_agg.groupby(['FTUE_flag','platform','days_since_install']).agg(
    total_users=('users', 'sum')
).reset_index()

pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(pl_ftue_funnel_total_agg, on=['FTUE_flag','platform','days_since_install'])
pl_ftue_funnel_agg['pctg_users'] = pl_ftue_funnel_agg['users'] / pl_ftue_funnel_agg['total_users']


pl_ftue_funnel_agg['pctg_diff_users'] = pl_ftue_funnel_agg.groupby(['max_level','platform','days_since_install'])['pctg_users'].pct_change().fillna(0)

pl_ftue_funnel_agg['combined_dimension'] = pl_ftue_funnel_agg['FTUE_flag'].astype(str) + ' | ' + pl_ftue_funnel_agg['days_since_install'].astype(str)

# Calculate percentiles by FTUE_flag, platform, and days_since_install
level_dist = data.groupby(['days_since_install', 'max_level', 'FTUE_flag', 'platform']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts_by_group = level_dist.groupby(['days_since_install', 'FTUE_flag', 'platform']).apply(
    weighted_quantiles, measure_col='max_level', include_groups=False
).reset_index()

# Rename columns for clarity
level_pcts_by_group = level_pcts_by_group.rename(columns={'p10': 'p10_max_level', 'p50': 'p50_max_level', 'p90': 'p90_max_level'})

# Merge into pl_ftue_funnel_agg
pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(
    level_pcts_by_group, 
    on=['days_since_install', 'FTUE_flag', 'platform'], 
    how='left'
)

pl_ftue_funnel_agg = pl_ftue_funnel_agg.loc[pl_ftue_funnel_agg['days_since_install'].isin([0,1,3,7,14,21])]


pl_ftue_funnel_agg

In [ ]:
# hide-output
# Define level milestone annotations for A.old and B.new FTUE feature unlock points
events_config = {
    'A.old': [
        {'level': 7, 'name': 'SP', 'color':'blue'},
        {'level': 8, 'name': 'Deco', 'color': 'blue'},
        {'level': 10, 'name': 'TimedC', 'color': 'blue'},
        {'level': 16, 'name': 'TA', 'color': 'blue'},
        {'level': 20, 'name': 'GenB', 'color': 'blue'},
        {'level': 25, 'name': 'TgtEvt', 'color': 'blue'},
    ],
    'B.new': [
        {'level': 6, 'name': 'SP', 'color': 'red'},
        {'level': 9, 'name': 'TASign', 'color': 'red'},
        {'level': 10, 'name': 'Deco', 'color': 'red'},
        {'level': 12, 'name': 'TA', 'color': 'red'},
        {'level': 14, 'name': 'GenB', 'color': 'red'},
        {'level': 17, 'name': 'TimedC', 'color': 'red'},
        {'level': 20, 'name': 'TgtEvt', 'color': 'red'},
    ]
}

In [ ]:
# Level distribution charts: user counts, percentage share, and day-over-day diff (up to level 30)
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='users',
              color='combined_dimension',
              title='Players level distribution',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)
#fig = add_median_lines(fig, pl_ftue_funnel_agg, x_col='p50_max_level', ftue_col='FTUE_flag', platform_col='platform')

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='pctg_users',
              color='combined_dimension',
              title='Players at each level (percentage)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='pctg_diff_users',
              color='combined_dimension',
              title='Players at each level (percentage diff change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)

fig.show()

### Percentile comparison

In [ ]:
# Band chart: P10/P50/P90 level progression comparison between A.old and B.new
plot_percentile_comparison(level_pcts_by_group, percentile='all')

# To inspect a single percentile with diff bar:
# plot_percentile_comparison(level_pcts_by_group, percentile='p50')
# plot_percentile_comparison(level_pcts_by_group, percentile='p10')
# plot_percentile_comparison(level_pcts_by_group, percentile='p90')

### Weighted average

For each install cohort, tracks the **weighted average max level** reached as a function of days since install. Weighted average is used to account for varying player counts across level buckets. The percentage-change chart below highlights where the steepest level gains occur in the early-day window.

In [ ]:
# hide-output
# Compute weighted average max level by FTUE group, platform, and days since install
days_since_install_baseline = 0

data_filtered = data[data['days_since_install'] >= days_since_install_baseline]

pl_ftue_max_level_agg = compute_weighted_progression(data_filtered, measure_col='max_level', dimension_cols=['dummy', 'days_since_install','FTUE_flag','platform'], min_bucket_size=50)
pl_ftue_max_level_agg['combined_dimension'] = pl_ftue_max_level_agg['dummy'].astype(str) + ' | ' + pl_ftue_max_level_agg['FTUE_flag']

pl_ftue_max_level_agg['pctg_diff_max_level'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['weighted_avg_max_level'].pct_change().fillna(0)

pl_ftue_total_users = pl_ftue_max_level_agg[pl_ftue_max_level_agg['days_since_install'] == 0][['FTUE_flag', 'platform', 'cohort_users']].rename(columns={'cohort_users': 'total_cohort_users'})
pl_ftue_max_level_agg = pl_ftue_max_level_agg.merge(pl_ftue_total_users, on=['FTUE_flag','platform'], how='left')
pl_ftue_max_level_agg['pctg_users'] = pl_ftue_max_level_agg['cohort_users'] / pl_ftue_max_level_agg['total_cohort_users']
pl_ftue_max_level_agg['pctg_diff_users'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['pctg_users'].pct_change().fillna(0)

pl_ftue_max_level_agg

In [ ]:
# hide-output
# Bar chart: surviving cohort size at each day since install by FTUE group
fig = px.bar(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='cohort_users',
              color='combined_dimension',
              title='Players at each day since install',
              facet_row='platform',
              width=1200,
              height=800,
              barmode='group',
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [ ]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='weighted_avg_max_level',
              color='combined_dimension',
              title='Player level reached at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

# hide-output
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='pctg_diff_max_level',
              color='combined_dimension',
              title='Player level reached at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [ ]:
# hide-output
# Sort data by user and day for per-user progression analysis
data.sort_values(['user_id', 'days_since_install'], inplace=True)
data

## Game day reached at day x (work in progress)

Mirrors the level analysis but uses **game days** (in-game calendar progression) instead of levels. Comparing both metrics reveals whether level gates or natural engagement drives pacing — if game days outpace levels, players are replaying content; if levels outpace game days, players are advancing quickly through fewer sessions.

In [ ]:
# hide-output
# Compute weighted average game day progression by install cohort and days since install

# Step 1: Count unique users per (install cohort, day since install, max_gameday bucket)
game_day_agg = data.groupby(['install_dt', 'days_since_install', 'max_gameday']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 2: Total unique users per cohort-day
daily_cohort_total_users = data.groupby(['install_dt','days_since_install']).agg(
    total_unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 3: Share of each day's users at each game day value
game_day_agg = game_day_agg.merge(daily_cohort_total_users, on=['install_dt', 'days_since_install'])
game_day_agg['percentage_of_daily_users'] = game_day_agg['unique_users'] / game_day_agg['total_unique_users']

# Step 4: Weighted average game day per cohort-day
weighted_avg = game_day_agg.groupby(['install_dt', 'days_since_install'], group_keys=False).apply(
    lambda x: (x['max_gameday'] * x['unique_users']).sum() / x['unique_users'].sum(),
    include_groups=False
).reset_index()

weighted_avg.columns = ['install_dt', 'days_since_install', 'weighted_avg_max_gameday']
game_day_agg = game_day_agg.merge(weighted_avg, on=['install_dt', 'days_since_install'])

# Drop small buckets (< 50 users) to reduce noise
game_day_agg = game_day_agg[game_day_agg['unique_users'] >= 50]

# Collapse to one row per cohort-day
game_day_agg = game_day_agg.groupby(['install_dt', 'days_since_install']).agg(
    cohort_users = ('unique_users', 'sum'),
    weighted_avg_max_gameday = ('weighted_avg_max_gameday', 'first')
).reset_index()

game_day_agg

In [ ]:
# hide-output
# Line chart: weighted average game day by install cohort
fig = px.line(game_day_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='install_dt',
              title='Game day daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},
              )


fig.show()

In [ ]:
# hide-output
# Compute and plot P10/P50/P90 game day distribution by days since install
gameday_dist = data.groupby(['days_since_install', 'max_gameday']).agg(
    users=('user_id', 'count')
).reset_index()

gameday_pcts = gameday_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_gameday', include_groups=False).reset_index()

fig = px.line(
    gameday_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_gameday'),
    x='days_since_install',
    y='max_gameday',
    color='percentile',
    title='Game day distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

In [ ]:
# Export notebook to HTML for sharing and archival
export_notebook_html(
    notebook_path='./earlyftue.ipynb',
    output_path='./earlyftue.html',
)